# Воркшоп №1 — Сбор мультимодального музыкального датасета

**Курсовая:** «Использование мультимодальных эмбеддингов в рекомендательных системах».

В этом ноутбуке я собираю датасет, где у каждого трека есть текст, обложка и аудио-превью.

Источники:
- **iTunes Search API** — метаданные, `previewUrl`, обложка
- **Last.fm API** — теги, wiki-описание, граф похожих треков

На выходе:
```
data/
├── metadata.parquet
├── audio/<track_id>.m4a
├── images/<track_id>.jpg
└── edges.parquet   # CF-граф из Last.fm getSimilar
```

Дальше этот датасет пойдёт в эмбеддинги, fusion и evaluation.

## 0. Установка зависимостей

In [1]:
%pip install -q requests pandas tqdm pyarrow tenacity python-slugify


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Конфигурация

Задаю пути к данным и HTTP-клиент. Ключ Last.fm беру из переменной окружения `LASTFM_API_KEY` (в git не коммичу). Без ключа парсинг тоже работает, но без тегов и wiki из Last.fm.

In [3]:
from __future__ import annotations

import os
import re
import json
import time
import hashlib
import logging
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Iterable

import requests
import pandas as pd
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

DATA_DIR = Path("data")
AUDIO_DIR = DATA_DIR / "audio"
IMAGE_DIR = DATA_DIR / "images"
META_PATH = DATA_DIR / "metadata.parquet"
RAW_DIR   = DATA_DIR / "raw"

for d in (AUDIO_DIR, IMAGE_DIR, RAW_DIR):
    d.mkdir(parents=True, exist_ok=True)

LASTFM_API_KEY = os.getenv("LASTFM_API_KEY", "")

ITUNES_SEARCH_URL  = "https://itunes.apple.com/search"
ITUNES_LOOKUP_URL  = "https://itunes.apple.com/lookup"
LASTFM_URL         = "https://ws.audioscrobbler.com/2.0/"

USER_AGENT = "CourseworkMultimodalRec/0.1 (research; contact: stokkozhin@edu.hse.ru)"

REQUEST_TIMEOUT = 20
SLEEP_BETWEEN   = 0.25

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("parser")
log.info("Storage prepared at %s", DATA_DIR.resolve())

2026-05-31 01:18:34,599 | INFO | Storage prepared at /Users/santokk/Documents/proga files/Курсач_мага/data


## 2. iTunes Search API

Через iTunes Search API ищу треки по жанрам и другим seed-запросам. Из ответа беру `previewUrl` (30-сек превью) и `artworkUrl100` (обложку, апскейлю до 600×600).

In [4]:
@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1, min=1, max=15),
    retry=retry_if_exception_type((requests.RequestException,)),
    reraise=True,
)
def _http_get(url: str, params: dict | None = None) -> requests.Response:
    resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
    if resp.status_code in (429, 500, 502, 503, 504):
        raise requests.RequestException(f"retryable status {resp.status_code}")
    resp.raise_for_status()
    return resp


def itunes_search(
    term: str,
    *,
    country: str = "US",
    limit: int = 200,
    media: str = "music",
    entity: str = "song",
) -> list[dict]:
    """Один запрос к iTunes Search API. Возвращает список треков (dict)."""
    params = {
        "term": term,
        "country": country,
        "media": media,
        "entity": entity,
        "limit": limit,
    }
    resp = _http_get(ITUNES_SEARCH_URL, params=params)
    data = resp.json()
    time.sleep(SLEEP_BETWEEN)
    return data.get("results", [])


def upscale_artwork(url: str, size: int = 600) -> str:
    """artworkUrl100 -> artworkUrl<size>x<size>bb."""
    if not url:
        return url
    return re.sub(r"/\d+x\d+bb\.(jpg|png)$", f"/{size}x{size}bb.jpg", url)


sample = itunes_search("queen rock", limit=3)
for t in sample:
    print(t["trackId"], "|", t["artistName"], "—", t["trackName"])
    print("   genre :", t.get("primaryGenreName"))
    print("   audio :", t.get("previewUrl"))
    print("   image :", upscale_artwork(t.get("artworkUrl100", "")))

1440651216 | Queen — We Will Rock You
   genre : Rock
   audio : https://audio-ssl.itunes.apple.com/itunes-assets/AudioPreview211/v4/e9/37/42/e9374231-9cef-ad56-365c-a7ba09e4fa55/mzaf_10566507321838390251.plus.aac.p.m4a
   image : https://is1-ssl.mzstatic.com/image/thumb/Music115/v4/4d/08/2a/4d082a9e-7898-1aa1-a02f-339810058d9e/14DMGIM05632.rgb.jpg/600x600bb.jpg
1434899938 | Queen — We Will Rock You (Movie Mix)
   genre : Rock
   audio : https://audio-ssl.itunes.apple.com/itunes-assets/AudioPreview211/v4/21/92/6e/21926ea9-e577-6152-e050-1f57d84d1810/mzaf_10071874708177001844.plus.aac.p.m4a
   image : https://is1-ssl.mzstatic.com/image/thumb/Music124/v4/ef/54/0b/ef540ba4-bbe7-0809-aa55-97ce32e0e13d/18UMGIM55763.rgb.jpg/600x600bb.jpg
1440628460 | Queen — We Will Rock You (Live At The Montreal Forum / November 1981)
   genre : Rock
   audio : https://audio-ssl.itunes.apple.com/itunes-assets/AudioPreview112/v4/e8/69/8a/e8698a3b-296e-5058-0652-a5ee89e059f4/mzaf_18378809766799338053.plus.aac

## 3. Last.fm

Last.fm нужен для двух вещей:
1. **Текст** — теги и wiki-описание трека (в iTunes текста мало)
2. **CF-сигнал** — позже соберу граф похожих треков через `track.getSimilar`

Использую метод `track.getInfo`. Если трек не найден — продолжаю только с полями из iTunes.

In [5]:
def _strip_html(text: str | None) -> str:
    if not text:
        return ""
    text = re.sub(r"<a [^>]*>.*?</a>", "", text, flags=re.S)
    text = re.sub(r"<[^>]+>", "", text)
    return text.strip()


def lastfm_track_info(artist: str, track: str) -> dict | None:
    """Last.fm track.getInfo."""
    if not LASTFM_API_KEY:
        return None
    params = {
        "method":  "track.getInfo",
        "api_key": LASTFM_API_KEY,
        "artist":  artist,
        "track":   track,
        "autocorrect": 1,
        "format":  "json",
    }
    try:
        resp = _http_get(LASTFM_URL, params=params)
        data = resp.json()
    except (requests.RequestException, ValueError) as e:
        log.warning("Last.fm getInfo fail for %s — %s: %s", artist, track, e)
        return None
    time.sleep(SLEEP_BETWEEN)
    if isinstance(data, dict) and "error" in data:
        log.info("Last.fm getInfo no data for %s — %s: %s", artist, track, data.get("message"))
        return None
    payload = (data or {}).get("track")
    if not payload:
        return None
    wiki = payload.get("wiki") or {}
    tags = [t["name"] for t in (payload.get("toptags") or {}).get("tag", []) if t.get("name")]
    return {
        "tags":      tags,
        "summary":   _strip_html(wiki.get("summary")),
        "content":   _strip_html(wiki.get("content")),
        "listeners": int(payload.get("listeners") or 0),
        "playcount": int(payload.get("playcount") or 0),
    }


if LASTFM_API_KEY:
    demo = lastfm_track_info("Queen", "Bohemian Rhapsody")
    print("tags:", demo["tags"][:8] if demo else None)
    print("summary:", (demo["summary"][:300] + "...") if demo and demo["summary"] else None)
else:
    print("LASTFM_API_KEY не задан — пропускаем демо. Можешь задать через os.environ['LASTFM_API_KEY'] = '...'")

tags: ['classic rock', 'rock', 'Queen', '70s', '80s']
summary: "Bohemian Rhapsody" is a song written by Freddie Mercury and originally recorded by the band Queen for their 1975 album A Night at the Opera.

The song is in the style of a stream-of-consciousness nightmare, and has unusual musical structure for popular music (it has no chorus, instead consisting of...


## 4. Скачивание превью и обложек

Скачиваю 30-сек аудио-превью и обложки 600×600. Файлы именую по `track_id`, уже скачанные пропускаю.

In [6]:
def _download_binary(url: str, dst: Path) -> bool:
    if dst.exists() and dst.stat().st_size > 0:
        return True
    try:
        with session.get(url, timeout=REQUEST_TIMEOUT, stream=True) as r:
            r.raise_for_status()
            tmp = dst.with_suffix(dst.suffix + ".part")
            with tmp.open("wb") as f:
                for chunk in r.iter_content(chunk_size=64 * 1024):
                    if chunk:
                        f.write(chunk)
            tmp.replace(dst)
        return True
    except (requests.RequestException, OSError) as e:
        log.warning("download fail %s -> %s: %s", url, dst.name, e)
        if dst.with_suffix(dst.suffix + ".part").exists():
            dst.with_suffix(dst.suffix + ".part").unlink(missing_ok=True)
        return False


def download_audio(track_id: int, preview_url: str) -> Path | None:
    if not preview_url:
        return None
    dst = AUDIO_DIR / f"{track_id}.m4a"
    return dst if _download_binary(preview_url, dst) else None


def download_cover(track_id: int, artwork_url: str, size: int = 600) -> Path | None:
    if not artwork_url:
        return None
    dst = IMAGE_DIR / f"{track_id}.jpg"
    return dst if _download_binary(upscale_artwork(artwork_url, size=size), dst) else None

## 5. Семена для парсинга

iTunes — поисковый API. Я иду по списку запросов: жанры, декады, mood-слова. Дубликаты убираю по `trackId`.

In [7]:
GENRES = [
    "rock", "pop", "hip hop", "rap", "r&b", "soul", "jazz", "blues",
    "electronic", "techno", "house", "drum and bass", "dubstep",
    "indie", "alternative", "metal", "punk", "folk", "country",
    "classical", "ambient", "lo-fi", "reggae", "latin", "k-pop",
]

DECADES = ["1970s", "1980s", "1990s", "2000s", "2010s", "2020s"]

MOODS = ["chill", "energetic", "sad", "romantic", "party", "workout", "study"]

SEED_QUERIES: list[str] = []
SEED_QUERIES += GENRES
SEED_QUERIES += [f"{g} {d}" for g in GENRES for d in DECADES]
SEED_QUERIES += [f"{m} music" for m in MOODS]

SEED_QUERIES = list(dict.fromkeys(SEED_QUERIES))
print(f"Всего seed-запросов: {len(SEED_QUERIES)}")
print("Примеры:", SEED_QUERIES[:5], "...", SEED_QUERIES[-5:])

Всего seed-запросов: 182
Примеры: ['rock', 'pop', 'hip hop', 'rap', 'r&b'] ... ['sad music', 'romantic music', 'party music', 'workout music', 'study music']


## 6. Сбор сырых метаданных iTunes

In [8]:
ITUNES_RAW_DIR = RAW_DIR / "itunes"
ITUNES_RAW_DIR.mkdir(parents=True, exist_ok=True)

raw_tracks: dict[int, dict] = {}

for query in tqdm(SEED_QUERIES, desc="iTunes seeds"):
    cache_path = ITUNES_RAW_DIR / (re.sub(r"[^a-z0-9]+", "_", query.lower()).strip("_") + ".json")
    if cache_path.exists():
        results = json.loads(cache_path.read_text())
    else:
        try:
            results = itunes_search(query, limit=200, country="US")
        except requests.RequestException as e:
            log.warning("Search '%s' failed: %s", query, e)
            results = []
        cache_path.write_text(json.dumps(results, ensure_ascii=False))

    for tr in results:
        tid = tr.get("trackId")
        if not tid or not tr.get("previewUrl"):
            continue
        if tid in raw_tracks:
            continue
        raw_tracks[tid] = {**tr, "_seed_query": query}

print(f"Уникальных треков с превью: {len(raw_tracks)}")

iTunes seeds:   0%|          | 0/182 [00:00<?, ?it/s]

Уникальных треков с превью: 16899


## 7. Обогащение Last.fm, скачивание медиа, сборка таблицы

In [9]:
MAX_TRACKS = None
DOWNLOAD_AUDIO  = True
DOWNLOAD_IMAGES = True
USE_LASTFM      = bool(LASTFM_API_KEY)

LASTFM_RAW_DIR = RAW_DIR / "lastfm"
LASTFM_RAW_DIR.mkdir(parents=True, exist_ok=True)

records: list[dict] = []
items = list(raw_tracks.items())
if MAX_TRACKS:
    items = items[:MAX_TRACKS]

for tid, tr in tqdm(items, desc="Tracks"):
    artist = tr.get("artistName", "")
    title  = tr.get("trackName", "")
    album  = tr.get("collectionName", "")

    audio_path: Path | None = None
    image_path: Path | None = None

    if DOWNLOAD_AUDIO:
        audio_path = download_audio(tid, tr.get("previewUrl", ""))
    if DOWNLOAD_IMAGES:
        image_path = download_cover(tid, tr.get("artworkUrl100", ""), size=600)

    lf: dict | None = None
    if USE_LASTFM and artist and title:
        lf_cache = LASTFM_RAW_DIR / f"{tid}.json"
        if lf_cache.exists():
            lf = json.loads(lf_cache.read_text())
        else:
            lf = lastfm_track_info(artist, title)
            lf_cache.write_text(json.dumps(lf, ensure_ascii=False))

    records.append({
        "track_id":      tid,
        "title":         title,
        "artist":        artist,
        "album":         album,
        "genre_itunes":  tr.get("primaryGenreName"),
        "release_date":  tr.get("releaseDate"),
        "duration_ms":   tr.get("trackTimeMillis"),
        "country":       tr.get("country"),
        "seed_query":    tr.get("_seed_query"),
        "preview_url":   tr.get("previewUrl"),
        "artwork_url":   upscale_artwork(tr.get("artworkUrl100", "")),
        "audio_path":    str(audio_path) if audio_path else None,
        "image_path":    str(image_path) if image_path else None,
        "lastfm_tags":       (lf or {}).get("tags") if lf else None,
        "lastfm_summary":    (lf or {}).get("summary") if lf else None,
        "lastfm_content":    (lf or {}).get("content") if lf else None,
        "lastfm_listeners":  (lf or {}).get("listeners") if lf else None,
        "lastfm_playcount":  (lf or {}).get("playcount") if lf else None,
    })

print(f"Собрано записей: {len(records)}")

Tracks:   0%|          | 0/16899 [00:00<?, ?it/s]

2026-05-31 01:49:24,792 | WARNING | download fail https://audio-ssl.itunes.apple.com/itunes-assets/AudioPreview221/v4/36/fd/35/36fd35e9-5c56-df35-1577-264f2c1360e2/mzaf_5814517274787560562.plus.aac.p.m4a -> 1434371882.m4a: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
2026-05-31 01:49:24,964 | WARNING | download fail https://is1-ssl.mzstatic.com/image/thumb/Music115/v4/b1/9f/ef/b19fef51-79de-a940-e8ab-9e4e07b04d96/18UMGIM53752.rgb.jpg/600x600bb.jpg -> 1434371882.jpg: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
2026-05-31 02:12:07,423 | WARNING | download fail https://audio-ssl.itunes.apple.com/itunes-assets/AudioPreview221/v4/e7/d6/6e/e7d66ee5-2ca5-7615-569a-cae0d2a34d99/mzaf_1473701202633846564.plus.aac.p.m4a -> 1450330681.m4a: HTTPSConnectionPool(host='audio-ssl.itunes.apple.com', port=443): Read timed out.
2026-05-31 02:12:07,632 | WARNING | download fail https://is1-ssl.mzstatic.com/imag

Собрано записей: 16899


## 8. Сохранение и проверка покрытия

In [10]:
df = pd.DataFrame.from_records(records)


def build_text_blob(row: pd.Series) -> str:
    parts = [
        f"{row['title']} — {row['artist']}",
        f"Album: {row['album']}" if row.get("album") else "",
        f"Genre: {row['genre_itunes']}" if row.get("genre_itunes") else "",
    ]
    tags = row.get("lastfm_tags")
    if isinstance(tags, list) and tags:
        parts.append("Tags: " + ", ".join(tags[:10]))
    if row.get("lastfm_summary"):
        parts.append(row["lastfm_summary"])
    return " \n".join(p for p in parts if p)


df["text_blob"] = df.apply(build_text_blob, axis=1)
df["has_audio"] = df["audio_path"].notna()
df["has_image"] = df["image_path"].notna()

df.to_parquet(META_PATH, index=False)
log.info("Saved metadata to %s (rows=%d)", META_PATH, len(df))

print("Покрытие модальностей:")
print(df[["has_audio", "has_image"]].mean().round(3))

print("\nТоп жанров iTunes:")
print(df["genre_itunes"].value_counts().head(15))

print("\nПример строки:")
print(df.iloc[0][["track_id", "title", "artist", "album", "genre_itunes",
                  "has_audio", "has_image", "text_blob"]].to_string())

2026-06-01 22:50:32,530 | INFO | Saved metadata to data/metadata.parquet (rows=16899)


Покрытие модальностей:
has_audio    0.876
has_image    0.876
dtype: float64

Топ жанров iTunes:
genre_itunes
Pop             2021
Rock            1520
Alternative     1292
Country         1145
R&B/Soul        1071
Electronic      1032
Hip-Hop/Rap      945
Dance            765
Classical        602
Reggae           517
K-Pop            375
Instrumental     335
Ambient          324
Hard Rock        324
Soundtrack       313
Name: count, dtype: int64

Пример строки:
track_id                                               1772783961
title                                                        Rock
artist                                                      Stepz
album                                               Rock - Single
genre_itunes                                          Hip-Hop/Rap
has_audio                                                    True
has_image                                                    True
text_blob       Rock — Stepz \nAlbum: Rock - Single \nGenre: H...


## 9. Коллаборативный граф (`track.getSimilar`)

Для бейзлайна нужен коллаборативный сигнал. Беру **Last.fm `track.getSimilar`** — граф похожих треков.

In [12]:
def lastfm_track_similar(artist: str, track: str, *, limit: int = 50) -> list[dict] | None:
    """Last.fm track.getSimilar. list — ответ API, None — ошибка."""
    if not LASTFM_API_KEY:
        return None
    params = {
        "method":      "track.getSimilar",
        "api_key":     LASTFM_API_KEY,
        "artist":      artist,
        "track":       track,
        "autocorrect": 1,
        "limit":       limit,
        "format":      "json",
    }
    try:
        resp = _http_get(LASTFM_URL, params=params)
        data = resp.json()
    except (requests.RequestException, ValueError) as e:
        log.warning("Last.fm getSimilar fail for %s — %s: %s", artist, track, e)
        return None
    time.sleep(SLEEP_BETWEEN)
    if not isinstance(data, dict):
        log.warning("Last.fm getSimilar unexpected payload for %s — %s: %r", artist, track, type(data))
        return None
    if "error" in data:
        log.info("Last.fm getSimilar no data for %s — %s: %s", artist, track, data.get("message"))
        return []
    similar = data.get("similartracks") or {}
    return similar.get("track", []) or []


SIMILAR_RAW_DIR = RAW_DIR / "lastfm_similar"
SIMILAR_RAW_DIR.mkdir(parents=True, exist_ok=True)

raw_edges: list[dict] = []
n_loaded = n_fetched = n_failed = 0

for tid, tr in tqdm(raw_tracks.items(), desc="track.getSimilar"):
    cache = SIMILAR_RAW_DIR / f"{tid}.json"
    if cache.exists():
        try:
            similar = json.loads(cache.read_text())
            n_loaded += 1
        except json.JSONDecodeError:
            cache.unlink(missing_ok=True)
            similar = None
    else:
        similar = None

    if similar is None and not cache.exists():
        fetched = lastfm_track_similar(tr["artistName"], tr["trackName"], limit=50)
        if fetched is None:
            n_failed += 1
            continue
        cache.write_text(json.dumps(fetched, ensure_ascii=False))
        similar = fetched
        n_fetched += 1

    if similar is None:
        continue

    for s in similar:
        raw_edges.append({
            "src_track_id": tid,
            "src_artist":   tr["artistName"],
            "src_title":    tr["trackName"],
            "dst_artist":   (s.get("artist") or {}).get("name"),
            "dst_title":    s.get("name"),
            "dst_mbid":     s.get("mbid") or None,
            "dst_playcount": int(s["playcount"]) if str(s.get("playcount") or "").isdigit() else None,
            "match":        float(s.get("match") or 0.0),
        })

print(f"Из кэша:           {n_loaded}")
print(f"Скачано в этот раз: {n_fetched}")
print(f"Сбойных (попробуем в следующий запуск): {n_failed}")
print(f"Всего сырых рёбер: {len(raw_edges)}")
print(f"Уникальных src треков: {len({e['src_track_id'] for e in raw_edges})}")

track.getSimilar:   0%|          | 0/16899 [00:00<?, ?it/s]

2026-06-02 01:01:12,628 | INFO | Last.fm getSimilar no data for Hunter Jordan — 1990s (feat. Cody Garrett): Track not found
2026-06-02 01:01:57,711 | INFO | Last.fm getSimilar no data for Chumina Power — Pop 2000's: Track not found
2026-06-02 01:04:09,528 | INFO | Last.fm getSimilar no data for MC Magic & Baby Bash — 2020: Track not found
2026-06-02 01:05:23,925 | INFO | Last.fm getSimilar no data for Ill Slim Collin — 1970s (feat. Grand Wizzard Theodore): Track not found
2026-06-02 01:09:07,197 | INFO | Last.fm getSimilar no data for Red Tattoo — Hip Hop 2000's: Track not found
2026-06-02 01:10:38,248 | INFO | Last.fm getSimilar no data for Sonic Vizion — 2020: Track not found
2026-06-02 01:11:31,673 | INFO | Last.fm getSimilar no data for BlackSkyº — Rap 1970: Track not found
2026-06-02 01:11:42,393 | INFO | Last.fm getSimilar no data for Sam Levine & Jack Jezzro — We've Only Just Begun: Track not found
2026-06-02 01:12:00,270 | INFO | Last.fm getSimilar no data for IA Productions — 

Из кэша:           5330
Скачано в этот раз: 11569
Сбойных (попробуем в следующий запуск): 0
Всего сырых рёбер: 476028
Уникальных src треков: 9602


### 9.1 Маппинг соседей в корпус

Соседей из Last.fm матчу с треками из датасета по artist + title. Получаю `edges.parquet`.

In [13]:
def _norm_key(s: str | None) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

key_to_id: dict[tuple[str, str], int] = {
    (_norm_key(r["artist"]), _norm_key(r["title"])): int(r["track_id"])
    for _, r in df.iterrows()
}

edges_df = pd.DataFrame(raw_edges)
edges_df["dst_track_id"] = edges_df.apply(
    lambda r: key_to_id.get((_norm_key(r["dst_artist"]), _norm_key(r["dst_title"]))),
    axis=1,
)
edges_df["in_corpus"] = edges_df["dst_track_id"].notna()

edges_df = edges_df.sort_values(["src_track_id", "match"], ascending=[True, False])
edges_df = edges_df.drop_duplicates(subset=["src_track_id", "dst_artist", "dst_title"])

EDGES_PATH = DATA_DIR / "edges.parquet"
edges_df.to_parquet(EDGES_PATH, index=False)

n_edges      = len(edges_df)
n_in_corpus  = int(edges_df["in_corpus"].sum())
src_with_any = edges_df.groupby("src_track_id")["in_corpus"].any().sum()
src_total    = edges_df["src_track_id"].nunique()

print(f"Всего рёбер:                  {n_edges}")
print(f"  из них с dst в нашем корпусе: {n_in_corpus} ({n_in_corpus / max(n_edges, 1):.1%})")
print(f"Src треков всего:             {src_total}")
print(f"  с >=1 внутрикорпусным соседом: {src_with_any} ({src_with_any / max(src_total, 1):.1%})")
print(f"Среднее число соседей на src:  {edges_df.groupby('src_track_id').size().mean():.1f}")
print(f"Сохранено в {EDGES_PATH}")

print("\nПример рёбер:")
print(
    edges_df.head(10)[["src_artist", "src_title", "dst_artist", "dst_title", "match", "in_corpus"]]
    .to_string(index=False)
)

Всего рёбер:                  476028
  из них с dst в нашем корпусе: 92999 (19.5%)
Src треков всего:             9602
  с >=1 внутрикорпусным соседом: 8385 (87.3%)
Среднее число соседей на src:  49.6
Сохранено в data/edges.parquet

Пример рёбер:
src_artist       src_title      dst_artist                                  dst_title    match  in_corpus
   Madonna Who's That Girl         Madonna                        Causing a Commotion 1.000000      False
   Madonna Who's That Girl         Madonna                               Dress You Up 0.763307      False
   Madonna Who's That Girl    Cyndi Lauper                            Time After Time 0.236193       True
   Madonna Who's That Girl   Kylie Minogue                       I Should Be So Lucky 0.232023      False
   Madonna Who's That Girl    Cyndi Lauper                                    She Bop 0.229082      False
   Madonna Who's That Girl    Taylor Dayne                        Tell It to My Heart 0.225328       True
   Madonna W

## 10. Итог и следующие шаги

Получил мультимодальный датасет и CF-граф. Дальше:
1. `02_embeddings.ipynb` — эмбеддинги E5 / CLIP / CLAP
2. `03_fusion.ipynb` — early / late / cross-attention fusion
3. `04_recsys_eval.ipynb` — warm vs cold evaluation
4. `05_compute.ipynb` — размер данных и время инференса